### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="naticusdroid_android_permissions_dataset",
    dataset_year="2021",
    domain_str="technology & internet",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/dataset/722/naticusdroid+android+permissions+dataset",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/naticusdroid_android_permissions_dataset/ && wget -P local-data-warehouse/naticusdroid_android_permissions_dataset/ https://archive.ics.uci.edu/static/public/722/naticusdroid+android+permissions+dataset.zip && unzip local-data-warehouse/naticusdroid_android_permissions_dataset/naticusdroid+android+permissions+dataset.zip -d local-data-warehouse/naticusdroid_android_permissions_dataset/  && rm local-data-warehouse/naticusdroid_android_permissions_dataset/naticusdroid+android+permissions+dataset.zip
""",
    # References
    academic_reference_bibtex="""@article{mathur2021naticusdroid,
  title={NATICUSdroid: A malware detection framework for Android using native and custom permissions},
  author={Mathur, Akshay and Podila, Laxmi Mounika and Kulkarni, Keyur and Niyaz, Quamar and Javaid, Ahmad Y},
  journal={Journal of Information Security and Applications},
  volume={58},
  pages={102696},
  year={2021},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="mathur2021naticusdroid",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- Originally, the data has 75% duplicates and a perfect class balance. The authors never mention duplicates in their paper. Given this large amount of duplicates, we dropped all row-duplicates, keeping only the first entry.
- We drop one duplicated column.
- Anomaly: all features are binary categorical features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Malware",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Malware",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/data.csv")

df = df.rename(
    columns={
        "Result": "Malware",
    }
)
df["Malware"] = df["Malware"].map({0: "No", 1: "Yes"})

# Drop duplicates
df = df.drop_duplicates(keep="first")

# Map all to cat
df = df.astype("category")

# Drop duplicated col
df = df.drop(columns=["me.everything.badger.permission.BADGE_COUNT_READ"])

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()